# Master Parquet file
## V_1.0.1

In [ ]:
# Import libraries and set up file paths
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../dataset/raw")
PROCESSED = Path("../dataset/processed")

In [ ]:
# Load the raw train, history, and complaints datasets
train = pd.read_csv(RAW / 'train.csv')
history = pd.read_csv(RAW / 'patient_history.csv')
complaints = pd.read_csv(RAW / 'chief_complaints.csv')

In [ ]:
# Remove duplicate columns and merge into one massive dataframe
history = history.drop(columns=[c for c in history.columns if c in train.columns and c != 'patient_id'])
complaints = complaints.drop(columns=[c for c in complaints.columns if c in train.columns and c != 'patient_id'])
complaints = complaints.drop(columns=[c for c in complaints.columns if c in history.columns and c != 'patient_id'])

merged = train.merge(history, on='patient_id', how='left').merge(complaints, on='patient_id', how='left')

In [ ]:
#ONE TIME RUN
csv_path = PROCESSED / "schema_preview.csv"

# Check if the file already exists
if not csv_path.exists():
    preview.to_csv(csv_path, index=False)
    print("File created.")
else:
    print("File already exists!")

Make the MASTER Parquet file


In [ ]:
# Save the massive combined file as a Parquet (much faster than CSV!)
parquet_path = PROCESSED / 'master_dataset.parquet'

if not parquet_path.exists():
    merged.to_parquet(parquet_path, index=False)
    print(f"Success! Master dataset saved to: {parquet_path}")
else:
    print(f"File already exists.")

print(f"Total size: {merged.shape[0]} patients, {merged.shape[1]} columns.")

In [ ]:
# Read the master file we just saved
master_df = pd.read_parquet(PROCESSED / 'master_dataset.parquet')

# Scan for missing values
missing_counts = master_df.isna().sum()

print("--- COLUMNS WITH MISSING DATA ---")
print(missing_counts[missing_counts > 0])

In [ ]:
# 1. Get the names of the text columns and number columns
categorical_cols = master_df.select_dtypes(include=[object, str]).columns
numeric_cols = master_df.select_dtypes(exclude=[object, str]).columns

# Step A: Count missing values for all text columns
# Step B: Filter down to ONLY the ones greater than 0
print("--- MISSING TEXT VALUES ---")
text_missing_counts = master_df[categorical_cols].isna().sum()
text_missing_only = text_missing_counts[text_missing_counts > 0]
print(text_missing_only)

#Do step A and step B for numeric_cols
print("\n--- MISSING NUMERIC VALUES ---")
number_missing_counts = master_df[numeric_cols].isna().sum()
number_missing_only = number_missing_counts[number_missing_counts > 0]
print(number_missing_only)


Make the CLEAN Parquet file

In [ ]:
clean_parquet_path = PROCESSED / 'clean_dataset.parquet'

if not clean_parquet_path.exists():
    # 1. List of 14 columns we decided to banish

    columns_to_drop = [
        # Non-clinical Identifiers
        'patient_id', 'site_id', 'triage_nurse_id', 
        # Time Logistics
        'arrival_hour', 'arrival_day', 'arrival_month', 'arrival_season', 'shift', 
        # Bias Variables
        'language', 'transport_origin', 'insurance_type', 
        # Data Leakage (The "Cheats")
        'disposition', 'ed_los_hours', 
        # Unstructured Text
        'chief_complaint_raw' 
    ]

    #23. Drop them all at once
    print(f"Original shape: {master_df.shape}")
    clean_df = master_df.drop(columns=columns_to_drop)
    print(f"Clean shape: {clean_df.shape}")

    clean_df.to_parquet(clean_parquet_path, index=False)
    print(f"\nSuccess! Clean dataset saved to: {clean_parquet_path}")
else:
    print(f"Clean dataset already exists!")
    clean_df = pd.read_parquet(clean_parquet_path)


In [ ]:
# 1. Create the feature dataset (X) by dropping the answer key
X = clean_df.drop(columns=['triage_acuity'])

# 2. Pull out the answer key (Y)
Y = clean_df['triage_acuity']

# 3. Find columns that contains text 
# We convert this to a simple python list() so CatBoost can read it
cat_features = list(X.select_dtypes(include=[object, str]).columns)

# Verify our work
print(f"Features (X) shape: {X.shape}")
print(f"Target (Y) shape: {Y.shape}")
print(f"\nFound {len(cat_features)} Categorical Features:")
print(cat_features)


Train Model

In [ ]:
import json
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# 1. Hide 20% of the data in a testing vault
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print(f"Training on {X_train.shape[0]} patients...")
print(f"Testing on hidden {X_test.shape[0]} patients...")

# 2. Load the parameters from the NEW v1.0.1.json file!
with open("../params/v1.0.1.json", "r") as f:
    config = json.load(f)
    cb_params = config["CATBOOST_PARAMS"]

print(f"\nLoaded CatBoost Parameters: {cb_params}")

# 3. Build the Base Model
model = CatBoostClassifier(**cb_params)

# 4. Train the AI!
print("\n--- INITIALIZING CATBOOST TRAINING ---")
model.fit(
    X_train, Y_train,
    cat_features=cat_features,  
    eval_set=(X_test, Y_test)   
)
print("--- TRAINING COMPLETE! ---")

# 5. SAVE THE MODEL TO THE HARD DRIVE
MODELS_DIR = Path("../models")
model_path = MODELS_DIR / "v1.0.1/model_v1.0.1.cbm"
model.save_model(str(model_path))
print(f"\nSUCCESS! Model physically saved to: {model_path}")


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

# 1. Ask the AI to take the final test
Y_pred = model.predict(X_test)

# 2.We compare its predictions (Y_pred) against the real answers (Y_test)
print("=== FINAL MODEL EVALUATION ===")
print(f"Overall Accuracy: {accuracy_score(Y_test, Y_pred) * 100:.2f}%\n")

print("--- Detailed Performance Breakdown ---")
# This prints Precision, Recall, and F1-Score for every single triage level!
print(classification_report(Y_test, Y_pred))

# 3. What was the AI thinking? (Feature Importance)
feature_importances = model.get_feature_importance()
feat_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=True).tail(15)

# Plot the top 15 most powerful clues
plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='skyblue', edgecolor='black')
plt.title("Top 15 Most Important Clinical Features (According to CatBoost)")
plt.xlabel("Importance Score")
plt.ylabel("Clinical Feature")
plt.tight_layout()
plt.show()
